# 06 - Train Wildfire Baseline Model

Baseline Spark MLlib model for `fire_occurred` using the cleaned Vietnam-only grid-day feature dataset.

Current verified run:

- Train: 2020-01-01 to 2023-12-31
- Test: 2024-01-01 to 2024-12-31
- Model: RandomForestClassifier, 100 trees, maxDepth=8, class weights enabled
- AUC-ROC: `0.8369`
- Precision: `0.5693`
- Recall: `0.7535`
- F1: `0.6486`

The model avoids label leakage by excluding same-day fire aggregate columns such as `fire_count`, `avg_frp`, `max_frp`, and `avg_fire_confidence` from the input features.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassificationModel, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [ ]:
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minioadmin")
MINIO_BUCKET = os.getenv("MINIO_BUCKET", "wildfire-data")

FEATURES_PATH = f"s3a://{MINIO_BUCKET}/features/"
MODEL_OUTPUT = f"s3a://{MINIO_BUCKET}/models/random_forest_fire_baseline/"
METRICS_OUTPUT = Path("../reports/model_metrics_week1.json")
IMPORTANCE_OUTPUT = Path("../reports/feature_importance_week1.csv")
IMPORTANCE_PLOT_OUTPUT = Path("../reports/feature_importance_week1.png")

TRAIN_END_DATE = "2023-12-31"
TEST_START_DATE = "2024-01-01"


In [ ]:
spark = (
    SparkSession.builder
    .appName("Wildfire RF Baseline")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(MINIO_ENDPOINT.startswith("https://")).lower())
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)


In [ ]:
df = spark.read.parquet(FEATURES_PATH)

month = F.month("date")
dayofyear = F.dayofyear("date")
df = (
    df.withColumn("month", month.cast("double"))
    .withColumn("dayofyear", dayofyear.cast("double"))
    .withColumn("month_sin", F.sin(2 * F.pi() * month / F.lit(12.0)))
    .withColumn("month_cos", F.cos(2 * F.pi() * month / F.lit(12.0)))
    .withColumn("dayofyear_sin", F.sin(2 * F.pi() * dayofyear / F.lit(366.0)))
    .withColumn("dayofyear_cos", F.cos(2 * F.pi() * dayofyear / F.lit(366.0)))
)

print(f"Rows: {df.count():,}")
df.printSchema()


In [ ]:
feature_cols = [
    "grid_lat_index",
    "grid_lon_index",
    "grid_lat",
    "grid_lon",
    "temperature_2m_max",
    "relative_humidity_2m_min",
    "wind_speed_10m_max",
    "precipitation_sum",
    "precipitation_sum_7days",
    "dry_days_count",
    "station_distance_km",
    "weather_points_count",
    "month_sin",
    "month_cos",
    "dayofyear_sin",
    "dayofyear_cos",
]

# Excluded to avoid leakage: fire_count, avg_fire_confidence, avg_frp, max_frp, fire_occurred, risk_level.
df = df.fillna(0.0, subset=feature_cols)
train_df = df.filter(F.col("date") <= F.to_date(F.lit(TRAIN_END_DATE)))
test_df = df.filter(F.col("date") >= F.to_date(F.lit(TEST_START_DATE)))

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")
train_df.groupBy("fire_occurred").count().orderBy("fire_occurred").show()
test_df.groupBy("fire_occurred").count().orderBy("fire_occurred").show()


In [ ]:
counts = {int(row["fire_occurred"]): int(row["count"]) for row in train_df.groupBy("fire_occurred").count().collect()}
negative_count = counts[0]
positive_count = counts[1]
total_count = negative_count + positive_count

negative_weight = total_count / (2.0 * negative_count)
positive_weight = total_count / (2.0 * positive_count)

train_df = train_df.withColumn(
    "class_weight",
    F.when(F.col("fire_occurred") == 1, F.lit(positive_weight)).otherwise(F.lit(negative_weight)),
)

class_weights = {
    "negative_count": negative_count,
    "positive_count": positive_count,
    "negative_weight": negative_weight,
    "positive_weight": positive_weight,
}
class_weights


In [ ]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="keep")
rf = RandomForestClassifier(
    labelCol="fire_occurred",
    featuresCol="features",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=8,
    seed=42,
)

pipeline = Pipeline(stages=[assembler, rf])
model = pipeline.fit(train_df)
predictions = model.transform(test_df).cache()


In [ ]:
auc = BinaryClassificationEvaluator(
    labelCol="fire_occurred",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
).evaluate(predictions)

matrix = {
    (int(row["fire_occurred"]), int(row["prediction"])): int(row["count"])
    for row in predictions.groupBy("fire_occurred", "prediction").count().collect()
}
tn = matrix.get((0, 0), 0)
fp = matrix.get((0, 1), 0)
fn = matrix.get((1, 0), 0)
tp = matrix.get((1, 1), 0)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
accuracy = (tp + tn) / (tp + tn + fp + fn)

metrics = {
    "auc_roc": auc,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "accuracy": accuracy,
    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp,
    "train_rows": train_df.count(),
    "test_rows": test_df.count(),
    "train_end_date": TRAIN_END_DATE,
    "test_start_date": TEST_START_DATE,
    "feature_columns": feature_cols,
    "class_weights": class_weights,
    "features_input": FEATURES_PATH,
    "model_output": MODEL_OUTPUT,
}

print(json.dumps(metrics, indent=2))


In [ ]:
model.write().overwrite().save(MODEL_OUTPUT)
METRICS_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT.write_text(json.dumps(metrics, indent=2, sort_keys=True) + "
", encoding="utf-8")
print(f"Saved model to {MODEL_OUTPUT}")
print(f"Saved metrics to {METRICS_OUTPUT}")


In [ ]:
rf_model = next(stage for stage in model.stages if isinstance(stage, RandomForestClassificationModel))
importance = pd.DataFrame(
    {"feature": feature_cols, "importance": rf_model.featureImportances.toArray().tolist()}
).sort_values("importance", ascending=False)

IMPORTANCE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
importance.to_csv(IMPORTANCE_OUTPUT, index=False)
importance


In [ ]:
plot_df = importance.sort_values("importance")
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(plot_df["feature"], plot_df["importance"], color="#2563eb")
ax.set_title("RandomForest Feature Importance")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(IMPORTANCE_PLOT_OUTPUT, dpi=160)
plt.show()
print(f"Saved plot to {IMPORTANCE_PLOT_OUTPUT}")


In [ ]:
spark.stop()
